In [1]:
from pathlib import Path
from time import perf_counter
from collections import Counter, defaultdict
import csv
import hashlib
import json
import math
import random
import re
import sys
import os
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
    SQLTransformer, RegexTokenizer, StopWordsRemover,
    CountVectorizer, CountVectorizerModel, IDF, Normalizer,
    NGram, HashingTF
)
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.functions import vector_to_array

SEED = 42
ROOT = Path.cwd()
WORK = ROOT / "practice7_data"
OUT = ROOT / "practice7_results"
WORK.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
SOURCE = ROOT / "sentiment_dataset.csv"
if not SOURCE.is_file():
    raise FileNotFoundError(f"Не найден файл: {SOURCE}")

spark = (SparkSession.builder.appName("TextAnalytics")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
LABEL_NAMES = {0: "Нейтральный", 1: "Положительный",
               2: "Отрицательный"}

In [2]:
from zipfile import ZipFile
from pathlib import PurePosixPath

LIT_SOURCE = ROOT / "Literature.zip"
LIT_JSON = WORK / "literature_raw.jsonl"
LIT_INVENTORY = WORK / "literature_inventory.csv"
LIT_REVIEW = WORK / "literature_review.csv"
ENCODING_OVERRIDE = {"poems/Pushkin/Руслан и Людмила.txt": "cp1251"}

def logical_zip_name(info):
    name = info.filename
    if not (info.flag_bits & 0x800):
        try:
            name = name.encode("cp437").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return name

inventory = []
with ZipFile(LIT_SOURCE) as archive, LIT_JSON.open(
        "w", encoding="utf-8", newline="\n") as dst:
    if sum(x.file_size for x in archive.infolist()) > 250_000_000:
        raise ValueError("Архив больше принятого учебного ограничения")
    for info in archive.infolist():
        name = logical_zip_name(info)
        path = PurePosixPath(name)
        if path.suffix.lower() != ".txt":
            continue
        if path.is_absolute() or ".." in path.parts or len(path.parts) != 3:
            raise ValueError(f"Неожиданный путь: {name}")
        if info.file_size > 8_000_000:
            raise ValueError(f"Слишком крупный файл: {name}")
        encoding = ENCODING_OVERRIDE.get(name, "utf-8-sig")
        text = archive.read(info).decode(encoding, errors="strict")
        document_id = hashlib.sha256(name.encode("utf-8")).hexdigest()
        folder, author, filename = path.parts
        work = re.sub(r"\.\s*Том\s+\d+$", "", path.stem)
        record = dict(document_id=document_id, folder=folder,
            author=author, filename=filename, text=text,
            work_group=f"{author}/{work}")
        dst.write(json.dumps(record, ensure_ascii=False) + "\n")
        inventory.append(dict(document_id=document_id, folder=folder,
            author=author, filename=filename, encoding=encoding,
            chars=len(text), work_group=record["work_group"],
            body_start=0, body_end=len(text), keep=True, reviewed=False))

manifest = pd.DataFrame(inventory).sort_values(["folder", "author", "filename"])
manifest.to_csv(LIT_INVENTORY, index=False)
if not LIT_REVIEW.exists():
    manifest.to_csv(LIT_REVIEW, index=False)
print("Текстовых файлов:", len(manifest))
print(manifest.groupby(["folder", "author"]).size().to_string())
print("Разметка границ и проверка произведений:", LIT_REVIEW)


Текстовых файлов: 373
folder     author    
poems      Blok          10
           Lermontov     18
           Nekrasov      15
           Pushkin       35
prose      Blok           4
           Bryusov       30
           Chekhov       77
           Dostoevsky    33
           Gogol         16
           Gorky         33
           Herzen        11
           Lermontov      3
           Pushkin       10
           Tolstoy       42
           Turgenev      10
publicism  Tolstoy       26
Разметка границ и проверка произведений: /content/practice7_data/literature_review.csv


In [3]:
AUTHORS = ["Chekhov", "Gogol", "Turgenev"]
review = spark.read.option("header", True).schema("""
        document_id STRING, folder STRING, author STRING,
        filename STRING, encoding STRING, chars LONG,
        work_group STRING, body_start INT, body_end INT,
        keep BOOLEAN, reviewed BOOLEAN
    """).csv(LIT_REVIEW.as_posix())
chosen = review.filter((F.col("folder") == "prose") &
        F.col("author").isin(AUTHORS) & F.col("keep"))

required = ["work_group", "body_start", "body_end", "chars"]
for field in required:
    assert chosen.filter(F.col(field).isNull()).count() == 0

literature = spark.read.schema("""
        document_id STRING, folder STRING, author STRING,
        filename STRING, text STRING, work_group STRING
    """).json(LIT_JSON.as_posix())

docs = chosen.join(literature.select("document_id", "text"), "document_id")
docs = docs.withColumn("body", F.expr(
        "substring(text, body_start + 1, body_end - body_start)"))
units = docs.select("author", "work_group").distinct()

w = Window.partitionBy("author").orderBy(F.sha2("work_group", 256))
wa = Window.partitionBy("author")
units = (units.withColumn("r", F.row_number().over(w))
        .withColumn("n", F.count("*").over(wa))
        .withColumn("split", F.when(F.col("r") <= F.floor(F.col("n") * .6), "train")
            .when(F.col("r") <= F.floor(F.col("n") * .8), "valid")
            .otherwise("test")).select("author", "work_group", "split"))
docs = docs.join(units, ["author", "work_group"])


word_tokenizer = RegexTokenizer(inputCol="body", outputCol="words",
        pattern=r"(?U)[^\W_]+", gaps=False, minTokenLength=1)
docs = word_tokenizer.transform(docs).filter(F.size("words") >= 300)
fragments = (docs.withColumn("start", F.explode(
        F.sequence(F.lit(1), F.size("words") - 299, F.lit(300))))
        .withColumn("fragment", F.expr("slice(words, start, 300)"))
        .withColumn("text_fragment", F.concat_ws(" ", "fragment")))
fragments.groupBy("author", "split").count().show()


+--------+-----+-----+
|  author|split|count|
+--------+-----+-----+
| Chekhov|train|  396|
| Chekhov|valid|   86|
|   Gogol|valid|   54|
|   Gogol| test|  793|
| Chekhov| test|  450|
|   Gogol|train|  329|
|Turgenev|train|  936|
|Turgenev|valid|  186|
|Turgenev| test|   91|
+--------+-----+-----+



In [4]:
fragments.show()

+-------+----------------+--------------------+------+------------+---------+-----+----------+--------+----+--------+--------------------+--------------------+-----+--------------------+-----+--------------------+--------------------+
| author|      work_group|         document_id|folder|    filename| encoding|chars|body_start|body_end|keep|reviewed|                text|                body|split|               words|start|            fragment|       text_fragment|
+-------+----------------+--------------------+------+------------+---------+-----+----------+--------+----+--------+--------------------+--------------------+-----+--------------------+-----+--------------------+--------------------+
|Chekhov|  Chekhov/Агафья|1a8de5bb82504daab...| prose|  Агафья.txt|utf-8-sig|18893|         0|   18893|true|   false|Антон Чехов\nАГАФ...|Антон Чехов\nАГАФ...|train|[антон, чехов, аг...|    1|[антон, чехов, аг...|антон чехов агафь...|
|Chekhov|  Chekhov/Агафья|1a8de5bb82504daab...| prose|  Агаф